# Document Ingestion Test Notebook

This notebook tests the document processing pipeline **BEFORE** storing to the database.

## Document Processing Pipeline

```
Load Document → Chunk Text → Enrich Metadata → Generate Embeddings → Store to DB
     ↓              ↓              ↓                    ↓
  (Step 1)      (Step 2)       (Step 3)           (Step 4)
```

**This notebook tests Steps 1-4 WITHOUT storing to database.**

### Supported Formats:
- **PDF**: Page-based extraction using PyPDFLoader
- **DOCX**: Paragraph-based with section/heading tracking
- **TXT**: Plain text split by paragraphs

## Setup

In [ ]:
import sys
sys.path.insert(0, '.')

from src.services.document_processor import get_document_processor
from src.services.embedding_service import get_embedding_service
from pprint import pprint
import json
from pathlib import Path

## Configuration

In [11]:
# Test Configuration
DOCUMENT_PATH = "eTMS.docx"  # ⚠️ CHANGE THIS to your document path
TENANT_ID = "3105b788-b5ff-4d56-88a9-532af4ab4ded"  # Your tenant ID

# Chunking parameters (default values)
CHUNK_SIZE = 1500
CHUNK_OVERLAP = 150

print(f"Document: {DOCUMENT_PATH}")
print(f"Tenant ID: {TENANT_ID}")
print(f"Chunk Size: {CHUNK_SIZE}")
print(f"Chunk Overlap: {CHUNK_OVERLAP}")

Document: eTMS.docx
Tenant ID: 3105b788-b5ff-4d56-88a9-532af4ab4ded
Chunk Size: 1500
Chunk Overlap: 150


## Initialize Services

In [12]:
print("=" * 60)
print("INITIALIZING SERVICES")
print("=" * 60)

# Get document processor
doc_processor = get_document_processor(
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP
)

# Get embedding service
embedding_service = get_embedding_service()

print(f"✅ Document Processor: {type(doc_processor).__name__}")
print(f"   - Chunk Size: {doc_processor.chunk_size}")
print(f"   - Chunk Overlap: {doc_processor.chunk_overlap}")
print(f"   - Overlap %: {(doc_processor.chunk_overlap/doc_processor.chunk_size)*100:.1f}%")
print()
print(f"✅ Embedding Service: {type(embedding_service).__name__}")
print(f"   - Model: {embedding_service.model_name}")
print(f"   - Dimension: {embedding_service.dimension}")

INITIALIZING SERVICES
✅ Document Processor: DocumentProcessor
   - Chunk Size: 1200
   - Chunk Overlap: 150
   - Overlap %: 12.5%

✅ Embedding Service: EmbeddingService
   - Model: all-MiniLM-L6-v2
   - Dimension: 384


## Step 1: Load Document

Load the document and extract raw content (no chunking yet).

In [13]:
print("=" * 60)
print("STEP 1: LOAD DOCUMENT")
print("=" * 60)

# Detect file type
file_ext = Path(DOCUMENT_PATH).suffix.lower()
print(f"File Type: {file_ext}")
print()

# Load based on file type
if file_ext == '.pdf':
    raw_documents = doc_processor.load_pdf(DOCUMENT_PATH)
    print(f"✅ Loaded PDF: {len(raw_documents)} pages")
elif file_ext in ['.docx', '.doc']:
    raw_documents = doc_processor.load_docx(DOCUMENT_PATH)
    print(f"✅ Loaded DOCX: {len(raw_documents)} paragraphs")
elif file_ext == '.txt':
    raw_documents = doc_processor.load_txt(DOCUMENT_PATH)
    print(f"✅ Loaded TXT: {len(raw_documents)} paragraphs")
else:
    raise ValueError(f"Unsupported file type: {file_ext}")

# Statistics
total_chars = sum(len(doc.page_content) for doc in raw_documents)
avg_chars = total_chars / len(raw_documents) if raw_documents else 0

print(f"Total Characters: {total_chars:,}")
print(f"Average Characters per Document: {avg_chars:.0f}")
print()

# Show first document
if raw_documents:
    print("First Document Preview:")
    print("-" * 40)
    first_doc = raw_documents[0]
    print(f"Content: {first_doc.page_content[:300]}...")
    print()
    print(f"Metadata:")
    pprint(first_doc.metadata)

STEP 1: LOAD DOCUMENT
File Type: .docx

{"docx_path": "eTMS.docx", "event": "loading_docx", "level": "info", "timestamp": "2025-11-27T02:35:54.212208Z"}
{"docx_path": "eTMS.docx", "paragraph_count": 4705, "heading_count": 155, "total_chars": 405483, "event": "docx_loaded_successfully", "level": "info", "timestamp": "2025-11-27T02:36:50.671217Z"}
✅ Loaded DOCX: 4705 paragraphs
Total Characters: 405,483
Average Characters per Document: 86

First Document Preview:
----------------------------------------
Content: TÀI LIỆU HỖ TRỢ ĐÀO TẠO VÀ HƯỚNG DẪN SỬ DỤNG PHẦN MỀM eTMS...

Metadata:
{'file_type': '.docx',
 'is_heading': False,
 'page': 1,
 'paragraph_index': 1,
 'section_number': None,
 'section_title': 'Unknown',
 'source': 'eTMS.docx',
 'style': 'Normal'}


## Step 2: Chunk Documents

Split documents into smaller chunks with overlap.

In [14]:
print("=" * 60)
print("STEP 2: CHUNK DOCUMENTS")
print("=" * 60)

# Chunk the documents
chunks = doc_processor.chunk_documents(raw_documents, add_chunk_metadata=True)

print(f"✅ Chunking Complete")
print(f"Original Documents: {len(raw_documents)}")
print(f"Total Chunks: {len(chunks)}")
print(f"Chunks per Document: {len(chunks) / len(raw_documents):.1f}")
print()

# Chunk statistics
chunk_lengths = [len(chunk.page_content) for chunk in chunks]
print(f"Chunk Statistics:")
print(f"  - Average Length: {sum(chunk_lengths) / len(chunk_lengths):.0f} chars")
print(f"  - Min Length: {min(chunk_lengths)} chars")
print(f"  - Max Length: {max(chunk_lengths)} chars")
print()

# Show first 3 chunks
print("First 3 Chunks:")
for i, chunk in enumerate(chunks[:3]):
    print(f"\n--- Chunk {i+1} ---")
    print(f"Length: {len(chunk.page_content)} chars")
    print(f"Chunk Index: {chunk.metadata.get('chunk_index')} / {chunk.metadata.get('chunk_total')}")
    if file_ext in ['.docx', '.doc']:
        print(f"Section: {chunk.metadata.get('section_title', 'N/A')}")
        print(f"Section Number: {chunk.metadata.get('section_number', 'N/A')}")
    print(f"Content: {chunk.page_content[:200]}...")
    print("-" * 40)

STEP 2: CHUNK DOCUMENTS
{"document_count": 4705, "chunk_size": 1200, "chunk_overlap": 150, "event": "chunking_documents", "level": "info", "timestamp": "2025-11-27T02:37:08.629171Z"}
{"original_document_count": 4705, "chunk_count": 4705, "avg_chunk_size": 86.18129649309246, "event": "documents_chunked_successfully", "level": "info", "timestamp": "2025-11-27T02:37:09.350809Z"}
✅ Chunking Complete
Original Documents: 4705
Total Chunks: 4705
Chunks per Document: 1.0

Chunk Statistics:
  - Average Length: 86 chars
  - Min Length: 1 chars
  - Max Length: 1192 chars

First 3 Chunks:

--- Chunk 1 ---
Length: 58 chars
Chunk Index: 0 / 4705
Section: Unknown
Section Number: None
Content: TÀI LIỆU HỖ TRỢ ĐÀO TẠO VÀ HƯỚNG DẪN SỬ DỤNG PHẦN MỀM eTMS...
----------------------------------------

--- Chunk 2 ---
Length: 33 chars
Chunk Index: 1 / 4705
Section: Unknown
Section Number: None
Content: Version 2.2 – February 02nd, 2024...
----------------------------------------

--- Chunk 3 ---
Length: 37 c

## Step 3: Enrich Metadata

Add tenant_id, timestamps, and custom metadata.

In [16]:
print("=" * 60)
print("STEP 3: ENRICH METADATA")
print("=" * 60)

# Add custom metadata
additional_metadata = {
    "document_name": Path(DOCUMENT_PATH).name,
    "original_filename": Path(DOCUMENT_PATH).name,
    "source_detail": "test_ingestion",
    "uploaded_by_admin": "test_user"
}

# Enrich metadata
enriched_chunks = doc_processor.enrich_metadata(
    chunks,
    tenant_id=TENANT_ID,
    additional_metadata=additional_metadata
)

print(f"✅ Metadata Enriched")
print(f"Total Chunks: {len(enriched_chunks)}")
print()

# Show enriched metadata for first chunk
print("First Chunk Metadata (Enriched):")
print("-" * 40)
pprint(enriched_chunks[0].metadata)
print()

# Verify tenant_id is present
has_tenant_id = all(chunk.metadata.get('tenant_id') == TENANT_ID for chunk in enriched_chunks)
print(f"✅ All chunks have correct tenant_id: {has_tenant_id}")

STEP 3: ENRICH METADATA
✅ Metadata Enriched
Total Chunks: 4705

First Chunk Metadata (Enriched):
----------------------------------------
{'chunk_index': 0,
 'chunk_total': 4705,
 'document_name': 'eTMS.docx',
 'file_type': '.docx',
 'ingested_at': '2025-11-27T02:37:58.229306',
 'is_heading': False,
 'original_filename': 'eTMS.docx',
 'page': 1,
 'paragraph_index': 1,
 'section_number': None,
 'section_title': 'Unknown',
 'source': 'eTMS.docx',
 'source_detail': 'test_ingestion',
 'style': 'Normal',
 'tenant_id': '3105b788-b5ff-4d56-88a9-532af4ab4ded',
 'uploaded_by_admin': 'test_user'}

✅ All chunks have correct tenant_id: True


## Step 4: Generate Embeddings (Sample)

Generate embeddings for first 3 chunks to verify the process.

In [17]:
print("=" * 60)
print("STEP 4: GENERATE EMBEDDINGS (Sample)")
print("=" * 60)
print("⚠️  Only generating embeddings for first 3 chunks (for testing)")
print()

sample_chunks = enriched_chunks[:3]

for i, chunk in enumerate(sample_chunks):
    print(f"\nChunk {i+1}:")
    print(f"  Text Preview: {chunk.page_content[:100]}...")
    
    # Generate embedding
    embedding = embedding_service.embed_query(chunk.page_content)
    
    print(f"  Embedding Dimension: {len(embedding)}")
    print(f"  First 10 values: {[f'{v:.4f}' for v in embedding[:10]]}")
    print(f"  Vector norm: {sum(v**2 for v in embedding)**0.5:.4f}")
    print("-" * 40)

print(f"\n✅ Embeddings generated successfully")

STEP 4: GENERATE EMBEDDINGS (Sample)
⚠️  Only generating embeddings for first 3 chunks (for testing)


Chunk 1:
  Text Preview: TÀI LIỆU HỖ TRỢ ĐÀO TẠO VÀ HƯỚNG DẪN SỬ DỤNG PHẦN MỀM eTMS...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

  Embedding Dimension: 384
  First 10 values: ['0.0363', '0.0486', '0.0081', '0.0043', '-0.0132', '-0.0375', '0.0788', '0.0275', '0.0749', '0.0532']
  Vector norm: 1.0000
----------------------------------------

Chunk 2:
  Text Preview: Version 2.2 – February 02nd, 2024...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

  Embedding Dimension: 384
  First 10 values: ['-0.0513', '0.0221', '0.0935', '-0.0388', '0.0099', '-0.0558', '-0.0500', '-0.0090', '-0.0596', '-0.0169']
  Vector norm: 1.0000
----------------------------------------

Chunk 3:
  Text Preview: https://forms.office.com/r/fBPq6PdHjr...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

  Embedding Dimension: 384
  First 10 values: ['-0.0569', '0.0378', '-0.0526', '0.0268', '-0.0418', '0.0989', '-0.0842', '-0.0011', '-0.0342', '-0.0354']
  Vector norm: 1.0000
----------------------------------------

✅ Embeddings generated successfully


## Step 5: Preview Database Format

Show what would be stored in the database (without actually storing).

In [18]:
print("=" * 60)
print("STEP 5: DATABASE STORAGE PREVIEW")
print("=" * 60)
print("This is what would be stored in langchain_pg_embedding table")
print()

# Show first 2 chunks in database format
for i, chunk in enumerate(enriched_chunks[:2]):
    print(f"\n--- Database Entry {i+1} ---")
    
    # This is the format stored in PostgreSQL
    db_entry = {
        "document": chunk.page_content,  # TEXT column
        "cmetadata": chunk.metadata,      # JSONB column
        "embedding": f"vector({embedding_service.dimension})"  # VECTOR column
    }
    
    print(json.dumps(db_entry, indent=2, ensure_ascii=False, default=str))
    print()

print(f"\n✅ Total entries ready for database: {len(enriched_chunks)}")

STEP 5: DATABASE STORAGE PREVIEW
This is what would be stored in langchain_pg_embedding table


--- Database Entry 1 ---
{
  "document": "TÀI LIỆU HỖ TRỢ ĐÀO TẠO VÀ HƯỚNG DẪN SỬ DỤNG PHẦN MỀM eTMS",
  "cmetadata": {
    "source": "eTMS.docx",
    "file_type": ".docx",
    "paragraph_index": 1,
    "section_title": "Unknown",
    "section_number": null,
    "page": 1,
    "is_heading": false,
    "style": "Normal",
    "chunk_index": 0,
    "chunk_total": 4705,
    "tenant_id": "3105b788-b5ff-4d56-88a9-532af4ab4ded",
    "ingested_at": "2025-11-27T02:37:58.229306",
    "document_name": "eTMS.docx",
    "original_filename": "eTMS.docx",
    "source_detail": "test_ingestion",
    "uploaded_by_admin": "test_user"
  },
  "embedding": "vector(384)"
}


--- Database Entry 2 ---
{
  "document": "Version 2.2 – February 02nd, 2024",
  "cmetadata": {
    "source": "eTMS.docx",
    "file_type": ".docx",
    "paragraph_index": 2,
    "section_title": "Unknown",
    "section_number": null,
    "page

## Step 6: Analyze Chunk Distribution

Analyze how chunks are distributed across sections (for DOCX files).

In [19]:
print("=" * 60)
print("STEP 6: CHUNK DISTRIBUTION ANALYSIS")
print("=" * 60)

if file_ext in ['.docx', '.doc']:
    # Count chunks per section
    section_counts = {}
    for chunk in enriched_chunks:
        section = chunk.metadata.get('section_title', 'Unknown')
        section_counts[section] = section_counts.get(section, 0) + 1
    
    print(f"Total Sections: {len(section_counts)}")
    print()
    
    # Show top 10 sections by chunk count
    print("Top 10 Sections by Chunk Count:")
    sorted_sections = sorted(section_counts.items(), key=lambda x: x[1], reverse=True)[:10]
    
    for section, count in sorted_sections:
        print(f"  {count:3d} chunks - {section[:60]}..." if len(section) > 60 else f"  {count:3d} chunks - {section}")
else:
    print(f"Section analysis only available for DOCX files")
    print(f"Current file type: {file_ext}")

STEP 6: CHUNK DISTRIBUTION ANALYSIS
Total Sections: 156

Top 10 Sections by Chunk Count:
  366 chunks - 4.10.2. LCL
  230 chunks - 5.5. Tạo đơn hàng dịch vụ thuê container, remooc.
  198 chunks - 4.11.1. Phụ phí/ Thu chi hộ FCL (FCL Surcharge/ Behalf)
  174 chunks - 4.10.1 FCL
  161 chunks - 2.3.4. Đăng Nhập Vào Hệ Thống
  110 chunks - 6.2 Quản lý SOA (giao diện mới)
  108 chunks - 7.2. Yêu Cầu Bảo Dưỡng Sửa chữa
  105 chunks - 4.2.2. LCL (LCL Buying)
  103 chunks - 3.3.1. Danh sách xe (Vehicle List)
  100 chunks - 3.2.2. Danh sách đối tác (Partner List)


## Step 7: Full Pipeline Test

Test the complete pipeline using `process_document()` method.

In [20]:
print("=" * 60)
print("STEP 7: FULL PIPELINE TEST")
print("=" * 60)
print("Testing the complete process_document() method")
print()

# Process document using the all-in-one method
processed_chunks = doc_processor.process_document(
    file_path=DOCUMENT_PATH,
    tenant_id=TENANT_ID,
    additional_metadata=additional_metadata
)

print(f"✅ Pipeline Complete")
print(f"Total Chunks: {len(processed_chunks)}")
print()

# Verify results match manual process
print("Verification:")
print(f"  Manual Process: {len(enriched_chunks)} chunks")
print(f"  Pipeline Method: {len(processed_chunks)} chunks")
print(f"  Match: {len(enriched_chunks) == len(processed_chunks)}")
print()

# Show sample chunk
print("Sample Chunk from Pipeline:")
print("-" * 40)
sample = processed_chunks[0]
print(f"Content: {sample.page_content[:150]}...")
print()
print("Metadata:")
pprint(sample.metadata)

STEP 7: FULL PIPELINE TEST
Testing the complete process_document() method

{"file_path": "eTMS.docx", "file_type": ".docx", "tenant_id": "3105b788-b5ff-4d56-88a9-532af4ab4ded", "event": "processing_document_started", "level": "info", "timestamp": "2025-11-27T02:41:46.299752Z"}
{"docx_path": "eTMS.docx", "event": "loading_docx", "level": "info", "timestamp": "2025-11-27T02:41:46.299752Z"}
{"docx_path": "eTMS.docx", "paragraph_count": 4705, "heading_count": 155, "total_chars": 405483, "event": "docx_loaded_successfully", "level": "info", "timestamp": "2025-11-27T02:41:57.643145Z"}
{"document_count": 4705, "chunk_size": 1200, "chunk_overlap": 150, "event": "chunking_documents", "level": "info", "timestamp": "2025-11-27T02:41:57.643145Z"}
{"original_document_count": 4705, "chunk_count": 4705, "avg_chunk_size": 86.18129649309246, "event": "documents_chunked_successfully", "level": "info", "timestamp": "2025-11-27T02:41:57.817971Z"}
{"file_path": "eTMS.docx", "file_type": ".docx", "tenant_id

## Summary

This notebook demonstrated the complete document ingestion pipeline:

### ✅ Steps Completed:

1. **Load Document** - Extracted raw content from file
2. **Chunk Documents** - Split into smaller pieces with overlap
3. **Enrich Metadata** - Added tenant_id, timestamps, custom fields
4. **Generate Embeddings** - Created vector representations (sample)
5. **Preview Database Format** - Showed storage structure
6. **Analyze Distribution** - Examined chunk distribution across sections
7. **Full Pipeline Test** - Verified end-to-end process

### 📊 Results:

- **Total Chunks**: See above
- **Chunk Size**: Configured size with overlap
- **Metadata**: Tenant isolation + section tracking
- **Embeddings**: 384-dimensional vectors ready for similarity search

### 🔄 Next Steps:

To actually store these chunks to the database, use:

```python
from src.services.rag_service import get_rag_service

rag_service = get_rag_service()
result = rag_service.ingest_document(
    tenant_id=TENANT_ID,
    file_path=DOCUMENT_PATH,
    additional_metadata=additional_metadata
)
```

### ⚠️ Important Notes:

- All chunks have `tenant_id` for isolation
- Section metadata preserved for DOCX files
- Embeddings are generated on-demand during storage
- Chunks are stored in `langchain_pg_embedding` table